# ESG Event-Driven Alpha Strategy - Quick Start (v3.3)

This notebook demonstrates the basic usage of the ESG Event-Driven Alpha Strategy.

**Version**: 3.3 (December 2025)
**Key Features**:
- Sentiment-first signal weights (45% intensity)
- Reddit API for social media data (primary source)
- Russell Midcap universe (172 stocks)
- 7-day holding period (research-optimized)

See [README.md](../README.md) for full documentation.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Import project modules
from src.data import PriceFetcher, FamaFrenchFactors
from src.data.reddit_fetcher import RedditFetcher  # Primary social media source
from src.nlp import ESGEventDetector, FinancialSentimentAnalyzer, ReactionFeatureExtractor
from src.signals import ESGSignalGenerator, PortfolioConstructor
from src.backtest import BacktestEngine, PerformanceAnalyzer, FactorAnalysis

print("✓ All modules imported successfully")

## Step 1: Fetch Price Data

In [ ]:
# Define parameters (using sample tickers from Russell Midcap)
tickers = ['AAPL', 'TSLA', 'XOM', 'JPM', 'MSFT', 'CVX', 'NKE', 'SBUX']
start_date = '2024-01-01'
end_date = '2024-12-31'

# Fetch price data (uses batch download with retry logic)
price_fetcher = PriceFetcher()
prices = price_fetcher.fetch_price_data(tickers, start_date, end_date)

print(f"✓ Fetched {len(prices)} rows of price data")
print(f"  Tickers: {prices.index.get_level_values('ticker').unique().tolist()}")
print(f"  Date range: {prices.index.get_level_values('Date').min()} to {prices.index.get_level_values('Date').max()}")
prices.head()

## Step 2: Detect ESG Events

In [ ]:
# Initialize event detector
event_detector = ESGEventDetector()

# Example ESG event texts
example_texts = [
    "AAPL announced major environmental fine from EPA for pollution violations.",
    "TSLA disclosed data breach affecting customer information.",
    "XOM facing discrimination lawsuit from employees."
]

# Detect events
for i, text in enumerate(example_texts):
    result = event_detector.detect_event(text)
    print(f"\nText {i+1}: {text[:60]}...")
    print(f"Event detected: {result['has_event']}")
    if result['has_event']:
        print(f"Category: {result['category_full']}")
        print(f"Confidence: {result['confidence']:.2f}")
        print(f"Keywords: {result['matched_keywords'][:3]}")

## Step 3: Analyze Sentiment

In [ ]:
# Initialize sentiment analyzer
sentiment_analyzer = FinancialSentimentAnalyzer()

# Example financial texts
financial_texts = [
    "Strong quarterly earnings beat expectations.",
    "Company facing major regulatory investigation.",
    "Product launch received mixed reviews."
]

# Analyze sentiment
sentiments = sentiment_analyzer.analyze_batch(financial_texts)

for text, sentiment in zip(financial_texts, sentiments):
    numeric_score = sentiment_analyzer.score_to_numeric(sentiment)
    print(f"\nText: {text}")
    print(f"Sentiment: {sentiment['label']} (score: {numeric_score:.3f})")

## Step 4: Generate Trading Signals

Using the **sentiment-first** signal weights (v3.3):
- **Intensity**: 45% (primary alpha driver)
- **Volume**: 25% (social volume indicates conviction)
- **Event Severity**: 20% (detection confidence is noisy)
- **Duration**: 10% (sentiment persistence)

In [ ]:
# Create mock events and reactions
feature_extractor = ReactionFeatureExtractor(sentiment_analyzer)

# Initialize signal generator with v3.3 sentiment-first weights
signal_generator = ESGSignalGenerator(
    weights={
        'event_severity': 0.20,  # Reduced: detection confidence is noisy
        'intensity': 0.45,       # INCREASED: sentiment is primary alpha driver
        'volume': 0.25,          # Social volume indicates conviction
        'duration': 0.10         # Persistence of sentiment momentum
    }
)

signals_list = []

for ticker in tickers[:3]:
    event_date = datetime.strptime(start_date, '%Y-%m-%d') + timedelta(days=30)
    
    # Mock event
    event_result = {
        'has_event': True,
        'category': 'E',
        'confidence': 0.8,
        'sentiment': 'negative'
    }
    
    # Mock social media reaction (simulating Reddit data)
    mock_tweets = feature_extractor.create_mock_social_data(
        ticker, event_date, n_tweets=100, sentiment_bias='negative'
    )
    
    reaction_features = feature_extractor.extract_features(mock_tweets, event_date)
    
    # Generate signal
    signal = signal_generator.compute_signal(
        ticker=ticker,
        date=event_date,
        event_features=event_result,
        reaction_features=reaction_features
    )
    
    signals_list.append(signal)

signals_df = pd.DataFrame(signals_list)
print("✓ Generated Trading Signals (sentiment-first weights):")
signals_df

## Step 5: Construct Portfolio

In [ ]:
# Construct portfolio
portfolio_constructor = PortfolioConstructor(strategy_type='long_short')
portfolio = portfolio_constructor.construct_portfolio(signals_df, method='quintile')

print("\nPortfolio Weights:")
print(portfolio)

# Portfolio statistics
stats = portfolio_constructor.get_portfolio_statistics(portfolio)
print("\nPortfolio Statistics:")
for key, value in stats.items():
    print(f"{key}: {value}")

## Step 6: Run Backtest

In [ ]:
# Run backtest with v3.3 parameters
backtest_engine = BacktestEngine(
    prices=prices,
    initial_capital=1000000.0,
    commission_pct=0.0005,
    slippage_bps=3
)

# Use 7-day holding period (research shows ESG alpha decays in 5-10 days)
results = backtest_engine.run(
    signals=portfolio,
    rebalance_freq='W',
    holding_period=7  # Reduced from 12 days in v3.3
)

print(f"✓ Backtest Complete")
print(f"  Final Portfolio Value: ${results.get_final_value():,.2f}")
print(f"  Total Return: {results.get_total_return()*100:.2f}%")

## Step 7: Analyze Performance

In [ ]:
# Generate performance tear sheet
perf_analyzer = PerformanceAnalyzer(results)
perf_analyzer.print_tear_sheet()

## Step 8: Factor Analysis (Proving Alpha)

In [ ]:
# Load Fama-French factors
ff_factors = FamaFrenchFactors()
factors = ff_factors.load_ff_factors(start_date, end_date, frequency='daily')

# Run factor regression
factor_analysis = FactorAnalysis()
factor_analysis.load_factors(factors)
regression_results = factor_analysis.run_regression(results.returns_series)

# Print interpretation
print(factor_analysis.interpret_results(regression_results))

## Visualization (Optional)

In [ ]:
import matplotlib.pyplot as plt

# Plot equity curve
equity_curve = results.get_equity_curve()

if not equity_curve.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(equity_curve.index, equity_curve.values)
    plt.title('Portfolio Equity Curve')
    plt.xlabel('Date')
    plt.ylabel('Portfolio Value ($)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No equity curve data available")

## Conclusion

This notebook demonstrated the complete pipeline (v3.3):

1. ✓ Data acquisition (batch price fetching with retry logic)
2. ✓ ESG event detection (rule-based with keyword dictionaries)
3. ✓ Sentiment analysis (FinBERT 70% + Lexicon 30%)
4. ✓ Signal generation (**sentiment-first weights**: 45% intensity)
5. ✓ Portfolio construction (long-short market-neutral)
6. ✓ Backtesting (7-day holding period)
7. ✓ Performance analysis (30+ institutional metrics)
8. ✓ Factor analysis (proving alpha)

### v3.3 Key Features
- **Sentiment-First Weights**: Intensity at 45% (primary alpha driver)
- **Reddit API**: Primary social data source with company name fallback
- **7-Day Holding**: Research shows ESG alpha decays in 5-10 days
- **Quality Filters**: Volume ≥2.5x, |sentiment| ≥0.1

### Next Steps
- Run production backtest: `python run_production.py --universe russell_midcap`
- See [docs/ACTION_ITEMS.md](../docs/ACTION_ITEMS.md) for deployment guide
- See [docs/CONFIGURATION_GUIDE.md](../docs/CONFIGURATION_GUIDE.md) for parameter tuning

### Documentation
- [README.md](../README.md) - Project overview
- [docs/ACTION_ITEMS.md](../docs/ACTION_ITEMS.md) - Setup & deployment
- [docs/NEXT_STEPS.md](../docs/NEXT_STEPS.md) - Action guide